# Scenario 01: Basic Inference, Embedding, and Provider Health Check

Validates that all configured env vars correspond to live, working endpoints:

- **Inference:** temperature=0.0 determinism, temperature=1.0 variation, sampling params accepted.
- **Embedding:** model returns vectors with expected dimension.
- **Providers:** inference, files, and vector_io providers are registered on the server.

## Setup

Load configuration from environment variables. Set via the test runner script or export manually.

In [ ]:
import os
import requests
from ogx_client import OgxClient
from scripts.helpers import response_text

base_url = os.environ.get("BASE_URL", "http://localhost:8321")
model = os.environ.get("INFERENCE_MODEL", "")
embedding_model = os.environ.get("EMBEDDING_MODEL", "")
embedding_dimension = int(os.environ.get("EMBEDDING_DIMENSION", "768"))
inference_provider = os.environ.get("INFERENCE_PROVIDER", "")
files_provider = os.environ.get("FILES_PROVIDER", "")
vector_io_provider = os.environ.get("VECTOR_IO_PROVIDER", "")

assert base_url, "BASE_URL must be set"
assert model, "INFERENCE_MODEL must be set"

## Deterministic behavior (temperature=0.0)

With temperature 0, the model should return the same output for the same prompt when called twice.


In [ ]:
client = OgxClient(base_url=base_url)

prompt = "Reply with exactly one number: 42."
r1 = client.responses.create(model=model, input=prompt, temperature=0.0)
r2 = client.responses.create(model=model, input=prompt, temperature=0.0)

assert r1.status == "completed", f"Expected status completed, got {r1.status}"
assert r2.status == "completed", f"Expected status completed, got {r2.status}"
text1 = response_text(r1)
text2 = response_text(r2)
assert text1 and text2, "Expected non-empty response text"
assert text1.strip() == text2.strip(), (
    f"Determinism failed: temp=0.0 should give identical outputs. Got: {text1!r} vs {text2!r}"
)

## Non-deterministic behavior (temperature=1.0)

With temperature 1.0, we only verify that the API completes and returns valid output; we do not require identical responses across runs.


In [ ]:
r = client.responses.create(
    model=model,
    input="Name one European capital city.",
    temperature=1.0,
)
assert r.status == "completed", f"Expected status completed, got {r.status}"
text = response_text(r)
assert text, "Expected non-empty response for temperature=1.0"

## Sampling parameter (temperature only)

Verify that temperature is accepted and a completed response is returned. Note: `top_p` is not supported in the Responses API `responses.create`; use the Chat Completions API for `top_p`.

In [ ]:
r = client.responses.create(
    model=model,
    input="Say hello in one word.",
    temperature=0.3,
)
assert r.status == "completed", f"Expected status completed, got {r.status}"
assert r.output is not None, "Expected output to be present"

## Embedding model health check

Verify the embedding model is reachable and returns vectors with the expected dimension. Uses the OpenAI-compatible `/v1/embeddings` endpoint.

In [ ]:
if not embedding_model:
    print("EMBEDDING_MODEL not set — skipping embedding test")
else:
    resp = requests.post(
        f"{base_url}/v1/embeddings",
        json={"model": embedding_model, "input": "health check"},
    )
    assert resp.status_code == 200, (
        f"Embeddings endpoint returned {resp.status_code}: {resp.text}"
    )

    data = resp.json()
    assert "data" in data, f"Expected 'data' key in response: {data}"
    assert len(data["data"]) > 0, "Expected at least one embedding"

    vector = data["data"][0]["embedding"]
    assert len(vector) == embedding_dimension, (
        f"Expected dimension {embedding_dimension}, got {len(vector)}"
    )

## Provider health check

Verify that all configured providers (inference, files, vector_io) are registered on the server.

In [ ]:
if not all([inference_provider, files_provider, vector_io_provider]):
    print("Provider vars not fully set — skipping provider check")
else:
    resp = requests.get(f"{base_url}/v1/providers")
    assert resp.status_code == 200, (
        f"Providers endpoint returned {resp.status_code}: {resp.text}"
    )

    providers = resp.json().get("data", [])
    provider_map = {}
    for p in providers:
        provider_map.setdefault(p["api"], []).append(p["provider_id"])

    for api, expected in [
        ("inference", inference_provider),
        ("files", files_provider),
        ("vector_io", vector_io_provider),
    ]:
        registered = provider_map.get(api, [])
        assert expected in registered, (
            f"{api} provider '{expected}' not registered. Available: {registered}"
        )

## QE Assertions summary

- Deterministic (temp=0.0): two identical prompts yield identical response text.
- Non-deterministic (temp=1.0): response completes with non-empty output.
- Temperature parameter: request completes successfully with output present.
- Embedding: model returns vectors with expected dimension.
- Providers: inference, files, and vector_io providers are registered on the server.